In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import from_origin

from gridr.chain.fft_filtering_chain import fft_filtering_oa_strip_chain
from gridr.core.convolution.fft_filtering import (
    fft_array_filter,
    fft_array_filter_output_shape,
)
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE = True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook  # enables plot interface in J notebook
    output_notebook()


def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.

    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing:
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape

    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)

    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)

    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x) ** 2 + (y_grid / sigma_y) ** 2))

    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum

    return x, y, kernel


mandrill_0 = mandrill[0]

# The `gaussian_kernel_2d` function implements the formula given in the first tutorial.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1), sigma=gaussian_sigma)

# Getting started: filtering a raster with the chain layer

This tutorial presents the simplest call to `fft_filtering_oa_strip_chain`, the entry
point of the chain layer. It performs the same filtering as the first core tutorial, this
time reading and writing rasterio datasets instead of NumPy arrays.

**What you'll learn**

- How to build a test raster in memory with ``rasterio.io.MemoryFile``
- How to size and open the output dataset from the filtering parameters
- How to call ``fft_filtering_oa_strip_chain`` and read its result back
- Which arguments the chain requires where the core layer provides a default

## Setting things up

We import the chain entry point, the core helper that predicts the output shape, and the
well known mandrill raster image used as test data throughout this guide.

In [ ]:
import numpy as np
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import from_origin

from gridr.chain.fft_filtering_chain import fft_filtering_oa_strip_chain
from gridr.core.convolution.fft_filtering import fft_array_filter_output_shape
from gridr.misc.mandrill import mandrill

mandrill_0 = mandrill[0]

The kernel is the normalized Gaussian of the first core tutorial, with
$\sigma_x = \sigma_y = 4$ and a 25x25 support. The chain takes it in the spatial domain,
exactly as the core does.

In [ ]:
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma * 6 + 1),
                                  sigma=gaussian_sigma)
kernel.shape

## Creating the input raster

The chain reads its input through rasterio, so it needs a dataset and not an array.
`rasterio.io.MemoryFile` provides one in memory, with the same GeoTIFF driver and the same
API as a file on disk, which keeps this notebook free of temporary files. A real workflow
differs only by the call that opens the dataset, `rasterio.open(path)` in place of
`MemoryFile().open()`.

A georeferencing is attached to the raster so that the discussion of the output transform
below has something to build on. The filtering itself never looks at it.

In [ ]:
profile_in = {
    "driver": "GTiff",
    "height": mandrill_0.shape[0],
    "width": mandrill_0.shape[1],
    "count": 1,
    "dtype": mandrill_0.dtype,
    "crs": "EPSG:32631",
    "transform": from_origin(400000.0, 5000000.0, 1.0, 1.0),
}

memfile_in = MemoryFile()
with memfile_in.open(**profile_in) as ds:
    ds.write(mandrill_0, 1)

ds_in = memfile_in.open()
ds_in.height, ds_in.width, ds_in.dtypes

## Sizing the output dataset

The chain writes into a dataset that is already open, and it never creates one. The
height and width of that dataset must match what the filtering produces. The chain raises
a `ValueError` when they do not, instead of writing a truncated raster.

That shape depends on the kernel, on the production window `win`, and on `boundary`,
`out_mode` and `zoom`, so it cannot be read off the input raster.
`fft_array_filter_output_shape` derives it from shapes alone, without touching a pixel.
The values passed to it have to be the ones passed to the chain, otherwise the check
fails at call time.

In [ ]:
boundary = "symmetric"
out_mode = "same"
zoom = 1

out_shape = fft_array_filter_output_shape(
    (ds_in.height, ds_in.width),
    kernel,
    win=None,
    boundary=boundary,
    out_mode=out_mode,
    zoom=zoom,
)
out_shape

With `out_mode="same"` the output covers exactly the production window, which defaults to
the whole raster, so the shape is that of the input. The other settings change it:
`"full"` adds the kernel support on every side, and a zoom of `1/Q` divides each axis by
`Q`.

Two fields of the output profile remain the caller's responsibility. The dtype is chosen
here as `float32`, which keeps the filtered values as the convolution produces them; an
integer product would instead pair an integer dtype with `round_out=True`. The transform
must follow the geometry of the output: unchanged for `out_mode="same"` with `win=None`
and `zoom=1`, translated by the first row and column of the window when one is given, and
scaled by `Q` when the output is decimated. With `out_mode="full"` the first output sample
sits ahead of the window by the kernel margin, so the origin has to be shifted by that
amount as well.

In [ ]:
profile_out = profile_in | {
    "height": out_shape[0],
    "width": out_shape[1],
    "dtype": "float32",
}

memfile_out = MemoryFile()
ds_out = memfile_out.open(**profile_out)

## Filtering the raster

The call mirrors the core one, with the dataset pair in place of the array, and with two
arguments that the chain does not default: `boundary` and `out_mode` must be passed
explicitly, where `fft_array_filter` defaults them to `"none"` and `"same"`.

`strip_size` is set to 128 rows here to exercise the strip loop. The default of 512 is
larger than half the height of this image, a case in which the chain falls back to a
single monolithic call.

`round_out` is disabled so that the output keeps the values produced by the convolution,
which the comparison below relies on.

In [ ]:
fft_filtering_oa_strip_chain(
    ds_in=ds_in,
    ds_out=ds_out,
    band=1,
    kernel=kernel,
    boundary=boundary,
    out_mode=out_mode,
    win=None,
    strip_size=128,
    zoom=zoom,
    round_out=False,
)

The chain returns `0` once every block has been written. Closing the output dataset
flushes it, after which the same memory file can be opened for reading.

In [ ]:
ds_out.close()

with memfile_out.open() as ds:
    filtered = ds.read(1)

filtered.dtype, filtered.shape

Input and output can now be compared side by side.

In [ ]:
plot_im({"input": mandrill_0, "output": filtered},
        prefix="chain_001_filtered_mandrill_0")

The strip loop reproduces the result of a single call on the whole array. The two follow
a different sequence of floating-point operations, with different FFT sizes and partial
sums, so the comparison needs a tolerance and not an exact equality.

In [ ]:
from gridr.core.convolution.fft_filtering import fft_array_filter

expected = fft_array_filter(
    mandrill_0, kernel, boundary=boundary, out_mode=out_mode
).data

np.testing.assert_allclose(filtered, expected, rtol=1e-5, atol=1e-4)

print(f"close within tolerance: {np.allclose(filtered, expected, rtol=1e-5, atol=1e-4)}")
print(f"bit for bit identical:  {np.array_equal(filtered, expected)}")

## What's next

You have now filtered your first raster through the chain layer. Sizing the output dataset
from the parameters is the step that has no equivalent in the core API, and it is also the
one that changes as soon as a production window, another output mode or a decimation
enters the call. The following tutorials cover those parameters and the effect of
`strip_size` on memory. The overview notebook lists which core arguments the chain
forwards, which it reinterprets, and which it refuses.